# LeetCode #1116: Print Zero Even Odd

https://leetcode.com/problems/print-zero-even-odd/

## Synchronization Approaches

| Approach | Mechanism | Notes |
| :--- | :--- | :--- |
| **Naive: Busy-Wait** | Spin loop on shared counter | Wastes CPU |
| **Optimal: Three Semaphores Coordinating ★** | `zeroSem(1)`, `evenSem(0)`, `oddSem(0)` | Zero always prints first; then routes to even/odd based on parity |

---

## Understanding the Methods

### Naive: Busy-Wait
Spin in a loop checking a shared flag. Works but burns CPU.

### Optimal: Three Semaphores Coordinating ★
Three semaphores gate access: `zeroSem` starts at 1 (zero prints first). After printing zero, it releases `oddSem` or `evenSem` based on whether the next number is odd or even. The odd/even thread prints its number and releases `zeroSem` to restart the cycle.

**Constraints:**
* Three threads call `zero()`, `even()`, `odd()` concurrently
* Output must be `0102030405...` for n=5
* `1 <= n <= 1000`


## Solutions
### C#

In [ ]:
using System.Threading;

public class ZeroEvenOdd
{
    private readonly int _n;
    private readonly SemaphoreSlim _zeroSem = new SemaphoreSlim(1, 1);
    private readonly SemaphoreSlim _evenSem = new SemaphoreSlim(0, 1);
    private readonly SemaphoreSlim _oddSem  = new SemaphoreSlim(0, 1);

    public ZeroEvenOdd(int n) => _n = n;

    public void Zero(Action<int> printNumber)
    {
        for (int i = 1; i <= _n; i++)
        {
            _zeroSem.Wait();
            printNumber(0);
            if (i % 2 == 1) _oddSem.Release();
            else            _evenSem.Release();
        }
    }

    public void Even(Action<int> printNumber)
    {
        for (int i = 2; i <= _n; i += 2)
        {
            _evenSem.Wait();
            printNumber(i);
            _zeroSem.Release();
        }
    }

    public void Odd(Action<int> printNumber)
    {
        for (int i = 1; i <= _n; i += 2)
        {
            _oddSem.Wait();
            printNumber(i);
            _zeroSem.Release();
        }
    }
}

### Python

In [ ]:
import threading

class ZeroEvenOdd:
    def __init__(self, n: int):
        self.n = n
        self.zero_sem = threading.Semaphore(1)
        self.even_sem = threading.Semaphore(0)
        self.odd_sem  = threading.Semaphore(0)

    def zero(self, printNumber) -> None:
        for i in range(1, self.n + 1):
            self.zero_sem.acquire()
            printNumber(0)
            if i % 2 == 1:
                self.odd_sem.release()
            else:
                self.even_sem.release()

    def even(self, printNumber) -> None:
        for i in range(2, self.n + 1, 2):
            self.even_sem.acquire()
            printNumber(i)
            self.zero_sem.release()

    def odd(self, printNumber) -> None:
        for i in range(1, self.n + 1, 2):
            self.odd_sem.acquire()
            printNumber(i)
            self.zero_sem.release()

### Go

In [ ]:
package main

type ZeroEvenOdd struct {
	n       int
	zeroGo  chan struct{}
	evenGo  chan struct{}
	oddGo   chan struct{}
}

func NewZeroEvenOdd(n int) *ZeroEvenOdd {
	z := &ZeroEvenOdd{
		n:      n,
		zeroGo: make(chan struct{}, 1),
		evenGo: make(chan struct{}, 1),
		oddGo:  make(chan struct{}, 1),
	}
	z.zeroGo <- struct{}{}
	return z
}

func (z *ZeroEvenOdd) Zero(printNumber func(int)) {
	for i := 1; i <= z.n; i++ {
		<-z.zeroGo
		printNumber(0)
		if i%2 == 1 {
			z.oddGo <- struct{}{}
		} else {
			z.evenGo <- struct{}{}
		}
	}
}

func (z *ZeroEvenOdd) Even(printNumber func(int)) {
	for i := 2; i <= z.n; i += 2 {
		<-z.evenGo
		printNumber(i)
		z.zeroGo <- struct{}{}
	}
}

func (z *ZeroEvenOdd) Odd(printNumber func(int)) {
	for i := 1; i <= z.n; i += 2 {
		<-z.oddGo
		printNumber(i)
		z.zeroGo <- struct{}{}
	}
}

### Rust

In [ ]:
use std::sync::{Arc, Mutex, Condvar};

// state: 0 = zero's turn, 1 = odd's turn, 2 = even's turn
struct ZeroEvenOdd {
    n: usize,
    state: Mutex<u8>,
    cv: Condvar,
}

impl ZeroEvenOdd {
    fn new(n: usize) -> Arc<Self> {
        Arc::new(ZeroEvenOdd { n, state: Mutex::new(0), cv: Condvar::new() })
    }

    fn zero(&self, print_number: impl Fn(i32)) {
        for i in 1..=(self.n as i32) {
            let mut s = self.state.lock().unwrap();
            while *s != 0 { s = self.cv.wait(s).unwrap(); }
            print_number(0);
            *s = if i % 2 == 1 { 1 } else { 2 };
            self.cv.notify_all();
        }
    }

    fn odd(&self, print_number: impl Fn(i32)) {
        for i in (1..=(self.n as i32)).step_by(2) {
            let mut s = self.state.lock().unwrap();
            while *s != 1 { s = self.cv.wait(s).unwrap(); }
            print_number(i);
            *s = 0;
            self.cv.notify_all();
        }
    }

    fn even(&self, print_number: impl Fn(i32)) {
        for i in (2..=(self.n as i32)).step_by(2) {
            let mut s = self.state.lock().unwrap();
            while *s != 2 { s = self.cv.wait(s).unwrap(); }
            print_number(i);
            *s = 0;
            self.cv.notify_all();
        }
    }
}

## Concurrency Scenarios

1. **Odd and Even threads start before Zero**: Both block on their semaphores (count = 0); Zero proceeds first when `zeroSem` is 1.
2. **n = 1 (only odd)**: Zero prints 0, releases `oddSem`; Even never acquires `evenSem` and exits its loop naturally.
3. **n = 2 (odd then even)**: Zero→0, Odd→1, Zero→0, Even→2 — exact interleaving enforced by semaphore hand-off.
4. **Even thread runs ahead**: After Zero signals `evenSem`, if Even is not scheduled yet, the buffered semaphore holds the signal — no data lost.
5. **High n (1000)**: 2000 zero-prints and 500 even + 500 odd prints, all strictly ordered through the three-semaphore protocol with zero busy-waiting.
